# Agrico Plant Disease CNN Training (TensorFlow)

This notebook downloads PlantVillage via TFDS, trains an EfficientNetB0 model, and saves the model + labels for the Agrico API.

**Output files:**
- `plantvillage_best.keras`
- `labels.json`
- `reports/` (classification report + confusion matrix)


## 1) Runtime Setup
Select **GPU** in Colab: Runtime ? Change runtime type ? Hardware accelerator ? GPU.


In [ ]:
!nvidia-smi -L || true
!python -V


## 2) Install Dependencies


In [ ]:
!pip -q install -U tensorflow tensorflow-datasets scikit-learn matplotlib


## 3) Download PlantVillage via TFDS


In [ ]:
import tensorflow_datasets as tfds
from pathlib import Path

DATA_DIR = Path('/content/tfds')
DATA_DIR.mkdir(parents=True, exist_ok=True)

builder = tfds.builder('plant_village', data_dir=str(DATA_DIR))
builder.download_and_prepare()
info = builder.info
print('Downloaded:', info.splits)
print('Classes:', len(info.features['label'].names))


## 4) Build Datasets
We use TFDS `train` split and create 80/10/10 train/val/test.


In [ ]:
import tensorflow as tf
import numpy as np

IMAGE_SIZE = 224
BATCH_SIZE = 32

(ds_train, ds_val, ds_test), info = tfds.load(
    'plant_village',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    as_supervised=True,
    with_info=True,
    data_dir=str(DATA_DIR),
)
class_names = list(info.features['label'].names)

def preprocess(image, label):
    image = tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    image = tf.cast(image, tf.float32)
    return image, label

AUTOTUNE = tf.data.AUTOTUNE
ds_train = ds_train.shuffle(2048).map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_val = ds_val.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_test = ds_test.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

print('Train batches:', tf.data.experimental.cardinality(ds_train).numpy())
print('Val batches:', tf.data.experimental.cardinality(ds_val).numpy())
print('Test batches:', tf.data.experimental.cardinality(ds_test).numpy())


## 5) Build Model (EfficientNetB0 Transfer Learning)


In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
], name='augmentation')

inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False, weights='imagenet', input_tensor=x, pooling='avg'
)
base_model.trainable = False
x = tf.keras.layers.Dropout(0.2)(base_model.output)
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs, name='plantvillage_effnetb0')

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])
model.summary()


## 6) Train and Save Best Model


In [ ]:
EPOCHS = 8

callbacks = [
    tf.keras.callbacks.ModelCheckpoint('plantvillage_best.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
]

history = model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS, callbacks=callbacks)
model.save('plantvillage_best.keras')

import json
with open('labels.json', 'w', encoding='utf-8') as f:
    json.dump(class_names, f, indent=2)


## 7) Evaluate (Classification Report + Confusion Matrix)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = []
y_pred = []
for batch_images, batch_labels in ds_test:
    preds = model.predict(batch_images, verbose=0)
    y_true.extend(batch_labels.numpy().tolist())
    y_pred.extend(np.argmax(preds, axis=1).tolist())

report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report)

cm = confusion_matrix(y_true, y_pred)
np.savetxt('confusion_matrix.csv', cm, delimiter=',', fmt='%d')
with open('classification_report.txt', 'w', encoding='utf-8') as f:
    f.write(report)


## 8) Export to Agrico
Download the files and place them in `d:/Agrico/python_api/models/`: 
- `plantvillage_best.keras`
- `labels.json`

Then restart the API and `/predict` will use the trained CNN.


In [ ]:
from google.colab import files
files.download('plantvillage_best.keras')
files.download('labels.json')
files.download('classification_report.txt')
files.download('confusion_matrix.csv')
